In [27]:
import polars as pl

from src.config import EXTERNAL_DATA_DIR

runs = EXTERNAL_DATA_DIR / "runs"


graphsage_df = pl.read_csv(runs / "200_trials.csv").with_columns(pl.lit(True).alias("graphsage"))
gcn_df = pl.read_csv(runs / "not_graphsage.csv")

columns = [
    "hidden_dim", 'Loss',
 'TEST: F1 Score',
 'TEST: ROC-AUC',
 "graphsage"
 ]

eda_df = pl.concat([graphsage_df.select(columns), gcn_df.select(columns)])
eda_df

hidden_dim,Loss,TEST: F1 Score,TEST: ROC-AUC,graphsage
i64,f64,f64,f64,bool
24,0.577685,0.876157,0.921132,true
24,0.57768,0.745567,0.828232,true
24,0.589175,0.835077,0.862661,true
24,0.57439,0.85759,0.943343,true
24,0.575336,0.797831,0.831702,true
…,…,…,…,…
24,0.63723,0.675703,0.783865,false
24,0.644087,0.690166,0.719152,false
24,0.641765,0.691781,0.756819,false


In [29]:
eda_df.group_by("hidden_dim").agg(pl.mean("Loss").alias("mean_loss"),
                                   pl.mean("TEST: F1 Score").alias("mean_f1_score"),
                                   pl.mean("TEST: ROC-AUC").alias("mean_roc_auc")
                                  )

hidden_dim,mean_loss,mean_f1_score,mean_roc_auc
i64,f64,f64,f64
24,0.620349,0.676246,0.794615
32,0.628922,0.632318,0.767667


In [30]:
eda_df.group_by("graphsage").agg(pl.mean("Loss").alias("mean_loss"),
                                   pl.mean("TEST: F1 Score").alias("mean_f1_score"),
                                   pl.mean("TEST: ROC-AUC").alias("mean_roc_auc")
                                  )

graphsage,mean_loss,mean_f1_score,mean_roc_auc
bool,f64,f64,f64
false,0.669039,0.489215,0.66734
true,0.580231,0.819348,0.894942


In [33]:
eda_df.sort("TEST: F1 Score", descending=True).head()

hidden_dim,Loss,TEST: F1 Score,TEST: ROC-AUC,graphsage
i64,f64,f64,f64,bool
24,0.569168,0.887461,0.96184,true
32,0.572214,0.887081,0.959696,true
24,0.581214,0.8865,0.942242,true
24,0.571334,0.88625,0.960819,true
32,0.598364,0.884672,0.947178,true


In [ ]:
import mlflow

all_runs = mlflow.search_runs(experiment_ids=["3", "4"])